In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from scipy import stats
import statsmodels.formula.api as smf
import statsmodels.api as sm
import warnings
warnings.filterwarnings("ignore")

# ── À adapter ────────────────────────────────────────────────
df = pd.read_csv("ton_fichier.csv")

COL_CLUSTER     = "cluster_label"      # colonne avec les labels HDBSCAN (-1, 0, 1, ...)
COL_CLUSTER_REF = 0                    # cluster de référence (le plus grand par ex.)

VARS_CONTINUES  = ["age", "duree_triage"]
VARS_BINAIRES   = ["sexe"]             # 0/1 déjà encodé
VARS_CATEG      = ["mode_arrivee", "signes_vitaux", "motif", "score_triage"]
# ─────────────────────────────────────────────────────────────

# ── Préparation ───────────────────────────────────────────────
df_model = df.copy()

# Renommer cluster -1 en "outliers" pour que statsmodels gère bien
df_model[COL_CLUSTER] = df_model[COL_CLUSTER].astype(str)
df_model[COL_CLUSTER] = df_model[COL_CLUSTER].replace("-1", "outliers")
ref = str(COL_CLUSTER_REF)

# Vérifier pas de NaN dans les variables du modèle
all_vars = VARS_CONTINUES + VARS_BINAIRES + VARS_CATEG
df_model = df_model.dropna(subset=all_vars + [COL_CLUSTER])
print(f"N après suppression NaN : {len(df_model):,}")
print(f"Distribution clusters :\n{df_model[COL_CLUSTER].value_counts().sort_index()}\n")

# ── Construction de la formule ────────────────────────────────
# Variables continues : directement
cont_terms = " + ".join(VARS_CONTINUES + VARS_BINAIRES)

# Variables catégorielles : C(var, Treatment(ref)) pour choisir la référence
# Tu peux spécifier la modalité de référence dans Treatment()
categ_terms = " + ".join([f"C({v})" for v in VARS_CATEG])

formula = f"{COL_CLUSTER} ~ {cont_terms} + {categ_terms}"
print(f"Formule : {formula}\n")

# ── Régression multinomiale ───────────────────────────────────
model = smf.mnlogit(formula, data=df_model).fit(
    method="newton",
    maxiter=300,
    disp=False,
)
print(model.summary())

# ── Extraction des odds ratios ────────────────────────────────
def extract_OR_table(model, ref_cluster):
    """
    Retourne un DataFrame avec OR, IC 95% et p-value
    pour chaque cluster vs référence.
    """
    rows = []
    clusters = [c for c in model.model.endog_names if c != ref_cluster]

    for i, cluster in enumerate(clusters):
        params  = model.params.iloc[:, i]
        bse     = model.bse.iloc[:, i]
        pvalues = model.pvalues.iloc[:, i]

        for var in params.index:
            if var == "Intercept":
                continue
            b   = params[var]
            se  = bse[var]
            pv  = pvalues[var]
            OR  = np.exp(b)
            lb  = np.exp(b - 1.96 * se)
            ub  = np.exp(b + 1.96 * se)
            rows.append({
                "cluster":   cluster,
                "variable":  var,
                "OR":        round(OR, 3),
                "IC_95_low": round(lb, 3),
                "IC_95_up":  round(ub, 3),
                "p_value":   round(pv, 4),
                "sign":      "***" if pv < 0.001 else "**" if pv < 0.01 else "*" if pv < 0.05 else "",
            })

    return pd.DataFrame(rows)

or_table = extract_OR_table(model, ref)
print("\n=== Odds Ratios (vs cluster de référence) ===")
print(or_table.to_string(index=False))

# ── Export ────────────────────────────────────────────────────
or_table.to_csv("or_multinomial_clusters.csv", index=False)

# ── Tableau pivot lisible par cluster ────────────────────────
print("\n=== OR significatifs (p < 0.05) par cluster ===")
sig = or_table[or_table["p_value"] < 0.05].copy()
sig["OR_format"] = sig.apply(
    lambda r: f"{r['OR']} [{r['IC_95_low']}–{r['IC_95_up']}] {r['sign']}", axis=1
)
pivot = sig.pivot(index="variable", columns="cluster", values="OR_format")
print(pivot.to_string())
pivot.to_csv("or_pivot_significatifs.csv")

# ── Pseudo R² ────────────────────────────────────────────────
print(f"\nPseudo R² de McFadden : {model.prsquared:.4f}")
print(f"Log-likelihood        : {model.llf:.1f}")
print(f"AIC                   : {model.aic:.1f}")

Le cluster de référence doit être le plus grand ou le plus "basal" cliniquement — par exemple le cluster des patients jeunes, non urgents, faible consommation. Tous les OR s'interprètent par rapport à lui.
Avec N=120 000 tout sera significatif comme pour Cramér's V — donc ici aussi tu regardes la magnitude de l'OR, pas juste le p-value. Un OR de 1.05 n'est pas intéressant même si p < 0.001. Concentre-toi sur les OR > 1.5 ou < 0.67 (effet modéré).